In [1]:
import os

# Change this variable if the root folder name has been changed
root_dir = "nvae-shape-encoding"
current_dir = os.getcwd()

if not current_dir.endswith(root_dir):
    %cd ../..

assert os.getcwd().endswith(root_dir)

/Users/freddy/Unshared/nvae-shape-encoding


/Users/freddy/Unshared/nvae-shape-encoding/venv/lib/python3.11/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [2]:
import lightning as L
import torch

from utils.const import SEED
from data_modules.acdc import ACDCMaskDataModule
from utils.utils import get_data, setup_device

# Setup device
device = setup_device()
print(f"Device: {device}")

# Seed
L.seed_everything(SEED)

# Load data
data_module = ACDCMaskDataModule(batch_size=20)

# Reseed after preprocessing data
L.seed_everything(SEED)

loader_train = data_module.train_dataloader()
loader_val = data_module.val_dataloader()
loader_test = data_module.test_dataloader()

data_train = get_data(loader_train)
data_val = get_data(loader_val)
data_test = get_data(loader_test)

data = torch.cat([data_train, data_val, data_test], dim=0)
print(data.shape)

num_samples = data.shape[0]


/Users/freddy/Unshared/nvae-shape-encoding/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Seed set to 1969


Device: mps
Preprocessed training data found. Loading...
Preprocessed test data found. Loading...


Seed set to 1969


torch.Size([2978, 4, 128, 128])


In [10]:
from utils.anatomical_validity_checker import AnatomicalValidityChecker

for data in [data_train, data_val, data_test]:
    num_valid = 0

    for slice in data:
        AV = AnatomicalValidityChecker(slice)
        if AV.count_violations() == 0:
            num_valid += 1

    print(f"% AV: {num_valid / len(data)}")

% AV: 0.9912331969608417
% AV: 1.0
% AV: 0.9972118959107806
